# Week 7a.2: Multi-Agent Collaboration

**The running example: a deep research agent over arXiv**

A research question over the literature does not always have one answer in one place. It typically involves several parts, each part has its own papers, and the papers have to be read before they can be cited. One agent can do that, slowly, in one context that fills up as it goes.

<img src="figures/team.png" width="700">

Two patterns with the same tools:

- **One agent** searches, reads, searches again, all in one context. 
- **A team** has a lead that writes sub-questions, workers that each take one in their own context and in parallel, and the lead again to put the findings together.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(model="nvidia/nemotron-3.5-super-vl-120b-a12b")
#model = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b")

## 1. The tools

Two tools in `research_tools.py`:

- `search_papers` takes a query and returns up to eight papers, each with its id in brackets. 
- `get_paper` takes one of a paper id and returns the full record. 


By default the papers come from `papers.json`, which contains the records for 244 papers fetched from arXiv around five research questions. Rate limits make it infeasible to retrieve the papers during class.

If you want to do a live search outside of the classroom, you should set `PAPER_SOURCE=live` in `.env` and the same two tools go to arXiv itself, or to Semantic Scholar if you have put an `S2_API_KEY` there.

In [ ]:
import research_tools as rt

papers = rt.cache()["papers"]
print(f"{len(papers)} papers in the cache\n")

example = papers["2503.13657"]
for key, value in example.items():
    print(f"  {key:16} {str(value)[:70]}")

A search over the file is a real search, not a lookup. It ranks the cached papers by word overlap with the query, so a well-aimed query finds the paper and a vague one finds neighbors.

In [ ]:
print(rt.search_papers("taxonomy of failure modes in multi-agent LLM systems", limit=4))

In [ ]:
print(rt.get_paper("2503.13657")[:700])

### Sample questions

Five research questions come with the file, each labeled with the papers an answer should cite.

In [ ]:
for question, spec in rt.questions().items():
    print(question)
    for g in spec["gold"]:
        print(f"    [{g}] {papers[g]['title'][:70]}")
    print()

## 2. Orchestrator and workers

<img src="figures/team.png" width="700">

An **orchestrator** reads the question and writes sub-questions. Each sub-question goes to a **worker**, which runs the assistant-tools loop from Week 4 with a narrower prompt and a fresh context every time it runs. 
- The delegation of tasks to workers by the orchestrator is known as **fan-out**

The workers run in **parallel**. Each returns a short finding, not its transcript. The orchestrator reads the findings and writes the report.
- The combination of the outputs of parallel operations into a single node is known as **fan-in**

In LangGraph this is a `map-reduce`: 
- a node that fans out with `Send`, one worker run per sub-question, and 
- a node that collects what they returned

In [ ]:
import operator
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class ResearchState(TypedDict):
    question: str
    sub_questions: list
    findings: Annotated[list, operator.add]     # one entry per worker, appended
    tokens: Annotated[int, operator.add]        # every node adds what it spent
    report: str


class WorkerState(TypedDict):
    #TODO

NOTE:

`findings` uses a reducer so that sub-agents can all write to the same context


### The Orchestrator
The orchestrator will serve as the lead and break down a research question into multiple questions for better coverage.

In [ ]:
class Plan(BaseModel):
    """The parts of a research question that need their own search."""
    sub_questions: list[str] = Field(
        description="Three sub-questions. Each covers a different part of the "
                    "question and is specific enough to search for on its own.")


PLAN_PROMPT = """A research question: {question}

Split it into three sub-questions that together cover it and do not overlap.
Every sub-question must stay on the question's own subject and reuse its key
terms, so that a literature search for the sub-question finds papers about
this question and not about a neighboring topic. Each sub-question must be
a standalone question without referring to other questions.

Cover: the main systems or methods that have been proposed, the problems or failures that have been
reported, and what has been measured or compared."""

planner = model.with_structured_output(Plan)


In [ ]:
question = "Why do multi-agent LLM systems fail"
planner.invoke(PLAN_PROMPT.format(question=question))

In [ ]:
def plan(state: ResearchState) -> dict:
    out = planner.invoke(PLAN_PROMPT.format(question=state["question"]))
    return {#TODO}

### The worker

A worker is a simple ReAct agent, consisting of an `assistant` node with the two paper tools bound

Need to define the worker agent template as a function, because every worker will be spawned at runtime with its own prompt and its own fresh context.
- It's common to simply use `create_agent` for simple ReAct agents used in multi-agent systems

In [ ]:
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display
import time

QUESTION = list(rt.questions())[0]
TOOLS = [rt.search_papers, rt.get_paper]


def build_agent(system_prompt):
    model_with_tools = model.bind_tools(TOOLS)
    sys_msg = SystemMessage(content=system_prompt)

    def assistant(state: MessagesState) -> dict:
        messages = model_with_tools.invoke([sys_msg] + state["messages"])
        return {"messages": [messages]}

    g = StateGraph(MessagesState)
    g.add_node("assistant", assistant)
    g.add_node("tools", ToolNode(TOOLS))
    g.add_edge(START, "assistant")
    g.add_conditional_edges("assistant", tools_condition)
    g.add_edge("tools", "assistant")
    return g.compile()

### Two helpers 
- One adds up the tokens the model reported 
- One lists the tool calls it made in order

In [ ]:
def tokens_of(messages):
    """Total tokens the model reported across a run."""
    return sum((m.usage_metadata or {}).get("total_tokens", 0)
               for m in messages if getattr(m, "usage_metadata", None))


def tool_calls_of(messages):
    """The tool calls in a run, in order, as (tool, first argument)."""
    calls = []
    for m in messages:
        for call in getattr(m, "tool_calls", None) or []:
            first = next(iter(call["args"].values()), "")
            calls.append((call["name"], str(first)[:60]))
    return calls


In [ ]:
WORKER_PROMPT = """You are a research assistant working on one sub-question of a
larger question. Start with a search that uses the sub-question's own words.
Read the 2-4 papers that matter with get_paper, preferring the papers
that introduced a system or method over papers that only apply or extend it.
Write a finding of at most 150 words: what those papers say about the
sub-question, with each claim followed by the paper id in brackets like
[2503.13657]. Cite only papers you have read."""

worker = build_agent(WORKER_PROMPT)        # the same loop, one job, a fresh context per run


In [ ]:
sub_question = [HumanMessage(content=QUESTION)]
result = worker.invoke({"messages": sub_question})
result

In [ ]:
def research(state: WorkerState) -> dict:
    #TODO
    messages = result["messages"]
    return {"findings": [(state["sub_question"], messages[-1].text)],
            "tokens": tokens_of(messages)}

IMPORTANT NOTE:

`research` takes a `WorkerState`, not the `ResearchState`. 
- Each worker sees one sub-question, so the entire `ResearchState` isn't needed.
- This is known as **context isolation**



### Dynamic Fan-Out
Involves spawning nodes at runtime
- Send API is LangGraph’s mechanism for dynamic fan-out

`Send("node", payload)`
- node: the name of the node to run
- payload: the input that a spawned node receives

`Send("research", {"sub_question": q})`
- spawns a copy of the `research` node
- passes in q into the `sub_question` field of WorkerState



In [ ]:
def fan_out(state: ResearchState):
    # fan out for each question in sub_questions
    return [#TODO]

The fan-out is a conditional edge that returns one `Send` per sub-question. Each `Send` starts the `research` node with its own input, and LangGraph runs them concurrently.

```python

graph.add_conditional_edges("plan", fan_out, ["research"])
```

- `plan`: the fan-out starts after this node runs
- `fan_out`: the fan-out method that'll spawn the sub-agent nodes
- `["research"]`:  The list of target nodes


### Fan-in: Synthesis
The orchestrator combines the outputs from the worker nodes and synthesizes them


In [ ]:
SYNTH_PROMPT = """Question: {question}

Findings from three researchers, each on one part of the question:

{findings}

Write the report: what is known, organized by the parts of the question, with
every claim followed by the paper id in brackets as the findings give it. Keep
every paper the findings cite; do not drop a citation when you condense, and
do not add papers the findings did not cite."""


def synthesize(state: ResearchState) -> dict:
    findings = "\n\n".join(f"Sub-question: {q}\n{f}" for q, f in state["findings"])
    out = model.invoke(SYNTH_PROMPT.format(question=state["question"], findings=findings))
    return {"report": out.text, "tokens": tokens_of([out])}

## 3. The Graph

In [ ]:
graph = StateGraph(ResearchState)
graph.add_node("plan", plan)
graph.add_node("research", research)
graph.add_node("synthesize", synthesize)

graph.add_edge(START, "plan")
#TODO add fan-out
graph.add_edge("research", "synthesize")
graph.add_edge("synthesize", END)

team = graph.compile()
display(Image(team.get_graph(xray=True).draw_mermaid_png()))

NOTE:

Three workers each making two or three model calls at the same moment, plus the plan and the synthesis, is a dozen calls in a short amount of time, and the Gemini free tier allows 15 calls/minute.

In [ ]:
def run_team(question):
    start = time.time()
    result = team.invoke({"question": question})
    return {"report": result["report"], "sub_questions": result["sub_questions"],
            "findings": result["findings"], "tokens": result["tokens"],
            "seconds": time.time() - start}

In [ ]:
multi = run_team(QUESTION)

print("sub-questions:")
for q in multi["sub_questions"]:
    print("  -", q)
print()
print(multi["report"])

In [ ]:
multi["findings"]

## 4. Read the trace

The report is the output. The findings are where the failures are.

In [ ]:
cited_by = {}
for q, finding in multi["findings"]:
    ids = rt.found_ids(finding)
    print(f"{q}")
    print(f"    cites {ids}\n")
    for i in ids:
        cited_by.setdefault(i, []).append(q[:40])

shared = {i: qs for i, qs in cited_by.items() if len(qs) > 1}
print("papers cited by more than one worker:", shared or "none")

NOTE:
- **A worker may answer a different question from the one it was given.** No verifier in the current system
    - The MAST taxonomy calls this *inter-agent misalignment*: the agents are each fine and the handoff between them is not.

- **Two workers may find the same papers.** The sub-questions may overlap without a checker, so two sub-agents may do the same work and the synthesis may get the same finding twice. 
    - That is a *specification* failure: the plan did not say what each worker should leave to the others.

In [ ]:
# write the report to a file
from pathlib import Path

Path("report.md").write_text(multi["report"])

## 5. Things to try

1. Change the number of sub-questions from three to two, and then to five. For each, record the tokens, the seconds, and how many distinct papers the findings cite. Where does an extra worker stop adding papers?

2. The synthesis prompt says "do not add papers the findings did not cite." Remove that sentence and run again. Compare the citations in the report against the ids in the findings. What did the lead add, and were they real?

3. Run the team on the other four questions in `rt.questions()`. For each, read the sub-questions the plan produced. Which plans overlap, and what did the overlap cost in tokens?

4. Write a sub-question by hand that two of the three workers would both go after, run it, and look at the findings. Then change the lead's prompt so the plan tells each worker what the others are doing. Does the overlap go away?

In [ ]:
# your experiments here